# Semantic repository retrieval

Goal: embed repository chunks and find source text with a similar meaning to a question. This is retrieval only; no chat LLM generates an answer.

This tutorial is standalone. On a new computer, first run the setup command in the README and select QAChat's `.venv` kernel.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Python environment: {sys.executable}")

## 1. Choose one compatible embedding model

Change `BACKEND` to switch engines. The next cell selects a matching example model. MLX and PyTorch use different weights. Hugging Face reuses its normal shared cache and downloads missing files when `ALLOW_DOWNLOAD` is true.

In [ ]:
BACKEND = "mlx"
ALLOW_DOWNLOAD = True

if BACKEND == "mlx":
    MODEL_ID = (
        "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ"
    )
else:
    MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

print(f"Backend: {BACKEND}")
print(f"Embedding model: {MODEL_ID}")

In [ ]:
from qachat.embeddings import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")

## 2. Prepare repository chunks

This repeats the required preparation so the tutorial does not depend on tutorial 50. The selected embedding tokenizer now supplies the real token count.

In [ ]:
from qachat.chunking import chunk_source_units
from qachat.repository import scan_repository
from qachat.source import extract_source_units, read_repository_files

report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)
source_files = read_repository_files(report)
units = extract_source_units(source_files)
chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Prepared {len(chunks)} chunks.")

## 3. Build an in-memory semantic index

Each chunk becomes one vector. The index is kept in memory in this layer; it is not saved to the repository.

In [ ]:
from qachat.retrieval import build_semantic_index

index = build_semantic_index(chunks, embedding_session)

print(f"Indexed chunks: {len(index.chunks)}")
print(f"Vector size: {len(index.embeddings[0])}")

## 4. Search by meaning

Try a question that does not need to contain the exact function name. The result is source evidence, not a generated answer.

In [ ]:
from qachat.retrieval import semantic_search

QUESTION = "How does the program choose MLX or PyTorch?"
results = semantic_search(index, QUESTION, count=5)

for result in results:
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    print(f"{result.score:.3f}  {location}  {chunk.symbol}")

In [ ]:
best = results[0]
print(best.chunk.text)

## 5. Release the model

The vectors and index remain ordinary Python data, but the model memory can now be released.

In [ ]:
embedding_session.close()
print("Embedding model released.")